In [ ]:
# ============================================================
# 🔥 PyTorch Multi-Model Training Pipeline (VGG, ResNet, etc.)
# ============================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

/home/ashish/anaconda3/envs/realesrgan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# ============================================================
# CONFIGURATION
# ============================================================
DATA_DIR_NORMAL = "./Dataset/Original"      # Folder with normal (noisy) images
DATA_DIR_DENOISED = "./Dataset/Denoised"  # Folder with denoised images
OUTPUT_DIR = "./trained_models_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-4

In [14]:
# ============================================================
# DEVICE CONFIGURATION
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"✅ Using {num_gpus} GPU(s): {torch.cuda.get_device_name(0) if num_gpus > 0 else 'CPU only'}")

✅ Using 4 GPU(s): NVIDIA GeForce RTX 3090


In [15]:
# ============================================================
# DATA TRANSFORMS
# ============================================================
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ]),
}

In [16]:
# ============================================================
# FUNCTION TO LOAD DATASETS
# ============================================================
def load_datasets(data_dir):
    dataset = datasets.ImageFolder(os.path.join(data_dir), transform=data_transforms['train'])
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    dataloaders = {
        'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
        'val': DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    }
    print(f"📁 Loaded dataset from {data_dir}: {len(train_ds)} train, {len(val_ds)} val")
    return dataloaders

In [28]:
import torch
import torch.nn as nn
import torchvision.models as models

def create_model(model_name, num_classes=1):
    model_name = model_name.lower()
    
    if model_name == "vgg16":
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        for param in model.features.parameters():
            param.requires_grad = False

        # Replace classifier to fit our task
        in_features = model.classifier[0].in_features  # Should be 25088
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    elif model_name == "mobilenetv2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for param in model.features.parameters():
            param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for param in model.parameters():
            param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    else:
        raise ValueError(f"❌ Unknown model name: {model_name}")

    # Convert to binary classifier (sigmoid output)
    model = nn.Sequential(
        model,
        nn.Sigmoid()
    )

    return model.to("cuda" if torch.cuda.is_available() else "cpu")


In [29]:
# ============================================================
# TRAINING FUNCTION
# ============================================================
def train_model(model, dataloaders, model_name, data_type):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=2)

    best_acc = 0.0
    save_path = os.path.join(OUTPUT_DIR, f"{model_name}_{data_type}_best.pth")

    for epoch in range(EPOCHS):
        print(f"\n🚀 Epoch {epoch+1}/{EPOCHS} — {model_name} ({data_type})")
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss, running_corrects = 0.0, 0
            for inputs, labels in tqdm(dataloaders[phase], desc=f"{phase}"):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    preds = torch.round(torch.sigmoid(outputs.squeeze()))
                    loss = criterion(outputs.squeeze(), labels.float())

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)
            print(f"{phase} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

            if phase == 'val':
                scheduler.step(epoch_loss)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), save_path)
                    print(f"💾 Saved best model: {save_path}")

    print(f"✅ Training complete for {model_name} ({data_type}). Best Acc: {best_acc:.4f}")
    return best_acc.item()


In [30]:
# ============================================================
# MAIN EXPERIMENT LOOP
# ============================================================
results = {}

for model_name in ["VGG16", "ResNet50", "MobileNetV2"]:
    for data_type, data_dir in [("normal", DATA_DIR_NORMAL), ("denoised", DATA_DIR_DENOISED)]:
        dataloaders = load_datasets(data_dir)
        model = create_model(model_name)
        acc = train_model(model, dataloaders, model_name, data_type)
        results[f"{model_name}_{data_type}"] = acc

# ============================================================
# FINAL COMPARISON
# ============================================================
print("\n📊 FINAL COMPARISON:")
print("=" * 40)
for model_name in ["VGG16", "ResNet50", "MobileNetV2"]:
    n_acc = results.get(f"{model_name}_normal", 0)
    d_acc = results.get(f"{model_name}_denoised", 0)
    print(f"{model_name}: Normal={n_acc:.4f} | Denoised={d_acc:.4f} | Δ={(d_acc - n_acc):.4f}")

📁 Loaded dataset from ./Dataset/Original: 1455 train, 364 val

🚀 Epoch 1/5 — VGG16 (normal)


train: 100%|██████████| 46/46 [00:03<00:00, 13.66it/s]


train Loss: 0.6652 | Acc: 0.4605


val: 100%|██████████| 12/12 [00:01<00:00,  8.55it/s]


val Loss: 0.6063 | Acc: 0.4368
💾 Saved best model: ./trained_models_comparison/VGG16_normal_best.pth

🚀 Epoch 2/5 — VGG16 (normal)


...

val Loss: 0.5381 | Acc: 0.7737

🚀 Epoch 5/5 — MobileNetV2 (denoised)


train: 100%|██████████| 18/18 [00:07<00:00,  2.30it/s]


train Loss: 0.5123 | Acc: 0.8040


val: 100%|██████████| 5/5 [00:02<00:00,  1.96it/s]

val Loss: 0.5373 | Acc: 0.7737
✅ Training complete for MobileNetV2 (denoised). Best Acc: 0.7737

📊 FINAL COMPARISON:
VGG16: Normal=0.4396 | Denoised=0.7883 | Δ=0.3488
ResNet50: Normal=0.4945 | Denoised=0.7810 | Δ=0.2865
MobileNetV2: Normal=0.4286 | Denoised=0.7737 | Δ=0.3452
